<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/sudoku/LLaDA_Sudoku_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LLaDA Sudoku on Colab Pro

Run the `adi` branch with a persistent model copy in Google Drive. Select **Runtime → Change runtime type → GPU → L4 or A100** (subject to availability), then run the cells in order.

The first session downloads approximately 16 GB of weights into this model's own Drive folder. Later sessions reuse that pinned snapshot. Each new runtime still loads the weights into GPU memory. The optional copy to `/content` speeds repeated model loads within a session.

For the private GitHub repository, authorize Colab to read GitHub or upload this notebook. The clone cell separately needs a fine-grained token with **Contents: read** permission: store it in Colab **Secrets** as `GITHUB_TOKEN` or enter it into the hidden prompt. Tokens are never stored in the clone URL or notebook source.

The Config Updater starts with one puzzle and inspectable MCTS initialization. Select `baseline` to use the original sampler. The separate iLLaDA notebook retains its baseline defaults. Evaluation is model-only: Sudoku rules validate answers, without repairing them.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from `adi`
Clone or update `adi`. A clean checkout on another branch can switch safely; local modifications and divergent commits are never discarded.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


## 2. Install LLaDA dependencies
Keep Colab's CUDA-enabled PyTorch. Inference always uses a fresh Python process, which sees the installed dependency versions. LLaDA retains Transformers 4.38.2.

If you switch model notebooks in the same runtime, run this installation cell again before inference. The two model profiles use different Transformers versions.


In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_DIR / "requirements/llada-colab.txt"),
    "-r", str(REPO_DIR / "requirements/dev.txt"),
], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


### Check the search on CUDA before loading LLaDA
The model forward pass, softmax, entropy, top-token filtering and confidence factors run on the **GPU**. Tree states, UCB bookkeeping, final sorting of compact action statistics, and notebook tables stay on the **CPU**. This keeps branch copies out of GPU memory.

Run these three small CUDA tests first. They compare CPU and CUDA results for positive, negative and zero-entropy rewards; verify cached completion; and check that every model input reaches CUDA. They use tiny controlled distributions, so they do not download weights. The subprocess explicitly requires CUDA and fails if it is unavailable.

The later inspection and generation cells use the real HF LLaDA model with `device="cuda"` and BF16 weights. Each subprocess loads one model, and the kernel only displays saved traces.


In [ ]:
cuda_checks = """
import torch
import pytest

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in the test subprocess. Select a GPU runtime.")
print("CUDA test device:", torch.cuda.get_device_name(0), flush=True)
raise SystemExit(pytest.main(["-q", "tests/decoding/test_mcts_cuda.py"]))
"""
subprocess.run([sys.executable, "-u", "-c", cuda_checks], cwd=REPO_DIR, check=True)


## 3. Mount Drive
Allow at least **20 GB free Drive storage per model snapshot**, plus dataset cache and experiment outputs. Downloading both models needs room for both copies. Colab Pro compute and Drive storage are separate allowances.

`COPY_MODEL_TO_RUNTIME = True` stages the persistent model on runtime disk. Set it to `False` to load directly from Drive. Existing experiment folders are not moved or deleted.


In [ ]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "llada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "Sudoku")


## 4. Load this run's base configuration
Set the model, cache locations, and timezone. Choose sample count and generation parameters in the **Config Updater** below, which writes the final configuration and checks the data before inference.


In [ ]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/sudoku/llada_mcts.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["experiment_root"] = str(DRIVE_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE
RUN_CONFIG = Path("/content/llada-sudoku-run.yaml")
RUNNER = REPO_DIR / "scripts/run_sudoku.py"
print("Model revision:", profile["revision"])


## 5. Cache the model on Drive
Download the pinned model's tokenizer, chat template, configuration, Python implementation, and all shards listed in its weight index. Later sessions reuse the same files; keep the model folder's hidden `.cache` directory.

Rerun this cell after an interrupted download. If validation reports missing files or insufficient disk space, resolve that before inference. No weights are committed to GitHub.


In [ ]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


## Config Updater
Start with one puzzle, 256 output positions, `Lc=2`, `C=3`, `K1=K2=2`, and a 32-forward search cap. `Lc=2` means two **output tokens** committed in a candidate; it does not mean two Sudoku cells. Token positions are zero-based and exclude the prompt.

All MCTS settings are editable below. `baseline` uses the original sampler; `confidence` greedily fills the root without search; `mcts_entropy` searches then completes the selected state. Confidence/MCTS require `steps = block_length = gen_length`, zero temperature, and `low_confidence`.

The folder name includes the actual method and settings. Each run gets its own local-time timestamp. The dataset's first preparation can process all one million rows even when selecting one puzzle.


In [ ]:
NUM_SAMPLES = 1
DECODING_METHOD = "mcts_entropy"  # baseline | confidence | mcts_entropy
config["data"]["num_samples"] = NUM_SAMPLES
config["generation"].update(
    gen_length=256,
    steps=256,
    block_length=256,
    temperature=0.0,
    remasking="low_confidence",
)
config["decoding"] = {
    "method": DECODING_METHOD,
    "seed": 1,
    "mcts": {
        "init_tokens": 2,       # Lc
        "num_candidates": 3,    # C
        "top_k_tokens": 2,      # K1
        "top_k_actions": 2,     # K2
        "max_model_calls": 32,  # includes the root, excludes completion
        "margin_gamma": 10.0,
        "exploration_c": 0.01,
        "epsilon": 1e-8,
    },
}

exp_name = experiment_paths.experiment_name(config["generation"], config["decoding"])
config["output_dir"] = str(experiment_paths.experiment_parent(config))
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(f"Experiment output directory: {config['output_dir']}\n")
print(RUN_CONFIG.read_text())
subprocess.run([sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG), "--dataset-only"], check=True)


## Inspect one expansion
Run this section with `DECODING_METHOD = "mcts_entropy"`. It loads the model in a subprocess and inspects the first puzzle. One cycle expands the root's two retained actions, evaluates each child separately, and backs up each reward. With the default settings this uses **three forwards: one root + two children**. Each child has one committed token and no greedy completion yet.

The reward is `(parent_entropy - child_entropy) / parent_entropy`, over remaining masked positions. `committed_position_entropy` is uncertainty removed simply by filling that position. `context_gain` measures the change at the other positions and may be negative. Entropy is in nats; confidence is a ranking score, not a normalized probability.


In [ ]:
import json
import uuid
import pandas as pd
from IPython.display import display


def read_trace(path):
    with Path(path).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


def show_mcts_cycle(events, cycle):
    current = [event for event in events if event.get("cycle") == cycle]
    if not current:
        raise ValueError(f"No cycle {cycle}; available: {sorted({e['cycle'] for e in events if 'cycle' in e})}")
    print(f"Cycle {cycle} | cumulative search forwards: {current[-1]['search_model_calls']}")
    selection = [event for event in current if event["stage"] == "selection"]
    print("1. Selection")
    for event in selection:
        print("Traversed node IDs:", event["path"])
        rows = [{"parent_id": decision["parent_id"], "chosen_id": decision["chosen_id"], **sibling}
                for decision in event["decisions"] for sibling in decision["siblings"]]
        if rows:
            display(pd.DataFrame(rows))
        else:
            print("Root is unexpanded: no sibling UCB comparison yet.")
    print("2. Expansion — each row is an independent child")
    display(pd.DataFrame([{"parent_id": e["parent_id"], "child_id": e["child_id"],
                           "depth": e["depth"], **e["action"]}
                          for e in current if e["stage"] == "expansion"]))
    print("3. One-step evaluation")
    evaluations = [e for e in current if e["stage"] == "evaluation"]
    columns = ["parent_id", "child_id", "entropy_before", "entropy_after",
               "committed_position_entropy", "reward", "context_gain", "search_model_calls"]
    display(pd.DataFrame([{key: e[key] for key in columns} for e in evaluations]))
    for e in evaluations:
        print(f"Child {e['child_id']}: {e['child_state']['partial_output']}")
    print("4. Backpropagation — child to root for each evaluated action")
    rows = []
    for event in current:
        if event["stage"] == "backpropagation":
            for update in event["updates"]:
                rows.append({"evaluated_child": event["child_id"], "ancestor_id": update["node_id"],
                             **{f"{key}_before": value for key, value in update["before"].items()},
                             **{f"{key}_after": value for key, value in update["after"].items()}})
    display(pd.DataFrame(rows))
    print("Per-position entropy (0 at committed positions)")
    if evaluations:
        display(pd.DataFrame({"position": range(len(evaluations[0]["entropies_before"])),
                              "parent": evaluations[0]["entropies_before"],
                              **{f"child_{e['child_id']}": e["entropies_after"] for e in evaluations}}))
    candidates = [e for e in current if e["stage"] == "candidate"]
    if candidates:
        print("Collected candidates (duplicate states do not increment the count)")
        display(pd.DataFrame(candidates))


In [ ]:
if config["decoding"]["method"] != "mcts_entropy":
    raise ValueError("Select mcts_entropy and rerun Config Updater for inspection.")
INSPECTION_DIR = DRIVE_ROOT / profile["label"] / "Sudoku" / "inspections" / uuid.uuid4().hex
INSPECTION_DIR.mkdir(parents=True, exist_ok=False)
INSPECTOR = REPO_DIR / "scripts/inspect_mcts.py"
ONE_STEP_TRACE = INSPECTION_DIR / "one_step.jsonl"
subprocess.run([
    sys.executable, "-u", str(INSPECTOR), "--phase", "one_step",
    "--config", str(RUN_CONFIG), "--trace-path", str(ONE_STEP_TRACE),
], check=True)
one_step_events = read_trace(ONE_STEP_TRACE)
assert torch.device(one_step_events[-1]["model_device"]).type == "cuda"
print("HF model device:", one_step_events[-1]["model_device"])
print("Search-state storage:", one_step_events[-1]["search_state_device"])
root_event = next(e for e in one_step_events if e["stage"] == "initialization")
print("Root retained actions — actual HF model predictions")
display(pd.DataFrame(root_event["actions"]))
show_mcts_cycle(one_step_events, 1)
print("Inspection summary:")
display(pd.DataFrame([one_step_events[-1]]).drop(columns=["selected_state"]))


## Inspect initialization
This fresh subprocess continues all MCTS cycles until it collects at least `C` distinct depth-`Lc` states, exhausts the frontier, or reaches the search-forward cap. Target-depth nodes are closed. A full cycle may overshoot `C` while evaluating its retained siblings.

Change only `CYCLE_TO_VIEW` to inspect another cycle without loading the model again. The final candidate is ranked by **root-to-candidate** normalized entropy reduction, rather than its last edge reward. If fewer than `C` candidates are reached, the trace records the fallback and selects the best state at the deepest evaluated depth.


In [ ]:
SEARCH_TRACE = INSPECTION_DIR / "search.jsonl"
subprocess.run([
    sys.executable, "-u", str(INSPECTOR), "--phase", "search",
    "--config", str(RUN_CONFIG), "--trace-path", str(SEARCH_TRACE),
], check=True)
search_events = read_trace(SEARCH_TRACE)
print("Available cycles:", sorted({e["cycle"] for e in search_events if "cycle" in e}))
summary = search_events[-1]
assert torch.device(summary["model_device"]).type == "cuda"
print("Selected partial state:", summary["selected_state"]["partial_output"])
display(pd.DataFrame([{k: v for k, v in summary.items() if k != "selected_state"}]))


In [ ]:
CYCLE_TO_VIEW = 1
show_mcts_cycle(search_events, CYCLE_TO_VIEW)


## 6. Run full generation from the selected state
This invocation loads the HF model once, repeats initialization with the saved settings, and **passes its selected partial state directly into completion**. It starts a fresh search; the inspection commands above are independent walkthroughs. The notebook kernel never loads the model.

For `mcts_entropy`, cached statistics supply the first completion commitment. Every later remaining token costs one completion forward. Inspect `search_model_calls` and `completion_model_calls` separately. Timings exclude loading; peak GPU memory is peak allocated bytes, including resident weights.

Each prediction, trace and plot is saved to a new Drive directory. `status: completed` means all artifacts were saved, not that Sudoku was solved. After changing settings, rerun Config Updater first.


In [ ]:
import json
import uuid

latest = None
# A unique receipt identifies this invocation; it cannot select an older run.
RUN_MANIFEST = Path("/content") / f"{MODEL_FAMILY}-run-{uuid.uuid4().hex}.json"
subprocess.run([
    sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG),
    "--run-manifest", str(RUN_MANIFEST),
], check=True)
receipt = json.loads(RUN_MANIFEST.read_text())
latest = Path(receipt["run_dir"])
print("This run:", latest)


## 7. Inspect all saved predictions and plots
All artifacts are in the current experiment's timestamp folder: `config.json`, `metrics.json`, `predictions.jsonl`, `traces/puzzle_000.jsonl`, and `plots/puzzle_000.png`, etc.

The displayed images are the PNGs saved during evaluation. Incomplete answers keep their original row positions, complete invalid grids are flagged, and red digits conflict with given clues. Nothing is repaired by a solver.


In [ ]:
if latest is None:
    raise RuntimeError("Complete the run cell before inspecting its results.")
print("Results:", latest)
print(json.dumps(json.loads((latest / "metrics.json").read_text()), indent=2))
print("\n--- All Predictions ---")
with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        record = json.loads(line)
        print(f"\nPrediction for Puzzle Index {record['index']}:")
        print(json.dumps(record, indent=2))


### Verify the actual generation trace on Colab
This checks the saved HF inference artifacts, forward accounting, and preservation of search commitments. It displays the raw prediction, existing Sudoku scores, and measured GPU cost. A failed Sudoku answer remains a valid decoder execution; quality is reported separately.


In [ ]:
records = [json.loads(line) for line in (latest / "predictions.jsonl").read_text().splitlines()]
assert len(records) == config["data"]["num_samples"]
PUZZLE_INDEX = 0
record = records[PUZZLE_INDEX]
stats = record["decoding_stats"]
print("Actual HF output:", record["raw_output"])
display(pd.DataFrame([{key: record[key] for key in ["parse_success", "clues_preserved", "valid_sudoku", "exact", "correct_cells"]}]))
display(pd.DataFrame([stats]))
assert torch.device(stats["model_device"]).type == "cuda", "The loaded HF model must be on CUDA."
assert stats["peak_gpu_memory_bytes"] > 0, "Run inference in the Colab CUDA runtime."
assert stats["total_model_calls"] == stats["search_model_calls"] + stats["completion_model_calls"]
if stats["method"] == "mcts_entropy":
    generation_events = read_trace(latest / record["trace_file"])
    evaluations = [e for e in generation_events if e["stage"] == "evaluation"]
    assert stats["search_model_calls"] == 1 + len(evaluations)
    assert stats["search_model_calls"] <= config["decoding"]["mcts"]["max_model_calls"]
    selected = next(e for e in generation_events if e["stage"] == "termination")["selected_state"]
    completions = [e for e in generation_events if e["stage"] == "completion"]
    positions = [c["output_position"] for c in selected["committed"]]
    positions += [e["action"]["output_position"] for e in completions]
    assert len(set(positions)) == len(positions) == config["generation"]["gen_length"]
    assert stats["completion_model_calls"] == max(0, len(completions) - 1)
    assert not completions or completions[-1]["remaining_masks"] == 0
    show_mcts_cycle(generation_events, globals().get("CYCLE_TO_VIEW", 1))
print("GPU artifact checks passed; see Sudoku quality metrics above and saved plots below.")


In [ ]:
from IPython.display import Image, display

with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        current_record = json.loads(line)
        plot_path = latest / current_record["plot_file"]
        print(f"Puzzle Index {current_record['index']} | {plot_path}")
        display(Image(filename=str(plot_path)))


## 8. Verify caching, then increase the sample count
After the one-puzzle verification, set `NUM_SAMPLES = 3` in the Config Updater and rerun it and the full-generation/results cells. To check reuse, rerun step 5 and evaluation: the pinned Drive snapshot and unchanged runtime files are reused, and a new experiment timestamp is created.

Review Sudoku validity and clue preservation before increasing to 10 or 100 puzzles. Change parameters only in the Config Updater; each run saves the resolved configuration. Old Drive result folders remain untouched.

### Notes and references
- Fixed-length masked generation uses the LLaDA checkpoint's own tokenizer and chat template. There is no training, solver repair, or automatic variable-length decoding.
- Exact accuracy compares all 81 digits with the reference. Cell accuracy divides matching cells by `81 × processed puzzles`. Failed parsing produces an empty prediction and zero matching cells.
- Completion, clue-preservation, and Sudoku-validity rates use processed puzzles as their denominator. A complete grid may still be invalid.
- Config metadata records the model identity/revision, Git commit, run time, and timezone. Saved notebook outputs are cleared in Git; run the cells to obtain new results.
- [Model](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct) · [Upstream sampler](https://github.com/ML-GSAI/LLaDA) · [Dataset](https://huggingface.co/datasets/certen/Sudoku-1M-Puzzles)
